# Preprocessing — Credit Card Fraud Detection

This notebook prepares the cleaned transaction dataset for supervised modeling.

**Objectives:**
- Perform a stratified three-way train/validation/test split
- Standardize numeric features using `StandardScaler`
- Address extreme class imbalance using SMOTE, applied to training data only
- Preserve original transaction amounts and timestamps needed for cost- and time-based analysis in later notebooks
- Save all preprocessed artifacts for downstream modeling

All transformations are learned strictly from the training data to prevent leakage into validation or test sets.

### 1. Imports

In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import os
import pickle

### 2. Environment Setup and Project Paths

In [33]:
# Absolute project root (one level up from notebooks/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Central processed data directory (consistent with other notebooks)
PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_PATH, exist_ok=True)

RANDOM_STATE = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_PATH:", PROCESSED_PATH)

PROJECT_ROOT: C:\Users\mail2\OneDrive\Desktop\UNSW\git_projects\fraud_detection
PROCESSED_PATH: C:\Users\mail2\OneDrive\Desktop\UNSW\git_projects\fraud_detection\data\processed


### 3. Loading clean data from 01_eda.ipnyb

In [34]:
def load_data(filepath=f"{PROCESSED_PATH}/eda_cleaned.csv"):
    """
    Load cleaned dataset from disk.
    No transformations are applied here.
    """
    df = pd.read_csv(filepath)
    print("Data loaded. Shape:", df.shape)
    return df

### Stratified Train / Validation / Test Split

A three-way split is used rather than a simple train/test split. Stratification preserves the dataset's extreme class imbalance (~0.17% fraud) across all three sets.

- **Train** is used to fit model parameters
- **Validation** is used for every model and threshold decision made in later notebooks (baseline model shortlisting, hyperparameter tuning, cost-sensitive threshold selection)
- **Test** is reserved exclusively for a single, final performance evaluation, and is not used for any decision-making prior to that point

In [35]:
def split_data(df, test_size=0.2, val_size=0.2, random_state=RANDOM_STATE):
    """
    Perform a stratified three-way split: train / validation / test.

    - train: used to fit model parameters
    - validation: used for all decisions (model shortlisting, tuning, threshold selection)
    - test: touched exactly once, for final reported results
    """
    X = df.drop("Target", axis=1)
    y = df["Target"]

    # First split off the test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Then split remaining data into train and validation
    # val_size is expressed as a fraction of the ORIGINAL full dataset,
    # so we recompute it as a fraction of X_temp
    relative_val_size = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=relative_val_size,
        stratify=y_temp,
        random_state=random_state
    )

    print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
    print("\nTrain Target Distribution:\n", y_train.value_counts())
    print("\nValidation Target Distribution:\n", y_val.value_counts())
    print("\nTest Target Distribution:\n", y_test.value_counts())

    return X_train, X_val, X_test, y_train, y_val, y_test

### Feature Scaling (StandardScaler)

Numeric features are standardized prior to oversampling and modeling:

- Scaling is required because SMOTE generates synthetic samples using distance metrics — without scaling, high-magnitude features like `Time` would dominate distance calculations, producing unrealistic synthetic points
- Distance- and gradient-based models (e.g. Logistic Regression) assume comparable feature scales
- The scaler is fit exclusively on training data and applied unchanged to validation and test, preventing leakage

In [36]:
def feature_scaling(X_train, X_val, X_test):
    """
    Scale features using StandardScaler.
    Fit only on training data; apply to validation and test.
    """
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    print("Scaling complete.")

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

### Handling Class Imbalance (SMOTE)

SMOTE oversampling is applied to the training set only. Validation and test sets retain their original, real-world class distribution, ensuring model evaluation reflects genuine deployment conditions rather than an artificially balanced one.

In [37]:
def handle_class_imbalance(X_train, y_train, random_state=RANDOM_STATE):
    """
    Apply SMOTE oversampling to training data.
    Validation and test sets keep their original class distribution.
    """
    sm = SMOTE(random_state=random_state)

    X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

    print("After SMOTE:\n", pd.Series(y_resampled).value_counts())

    return X_resampled, y_resampled

### 7. Saving preprocessed data

In [38]:
def save_preprocessed_data(X_train, X_val, X_test, y_train, y_val, y_test, scaler):
    """
    Save preprocessed datasets and fitted scaler to disk.
    """
    pd.DataFrame(X_train).to_csv(f"{PROCESSED_PATH}/X_train.csv", index=False)
    pd.DataFrame(X_val).to_csv(f"{PROCESSED_PATH}/X_val.csv", index=False)
    pd.DataFrame(X_test).to_csv(f"{PROCESSED_PATH}/X_test.csv", index=False)

    pd.DataFrame(y_train, columns=["Target"]).to_csv(f"{PROCESSED_PATH}/y_train.csv", index=False)
    pd.DataFrame(y_val, columns=["Target"]).to_csv(f"{PROCESSED_PATH}/y_val.csv", index=False)
    pd.DataFrame(y_test, columns=["Target"]).to_csv(f"{PROCESSED_PATH}/y_test.csv", index=False)

    with open(f"{PROCESSED_PATH}/scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    print("All preprocessed data (train/val/test) and scaler saved successfully.")

## Pipeline Execution

The preprocessing steps are executed in sequence: load the cleaned dataset, perform the stratified split, preserve original (unscaled) transaction values needed for later analysis, apply scaling, apply SMOTE to the training set, and save all resulting artifacts.

### Load Cleaned Dataset

The minimally cleaned dataset produced in the EDA notebook is loaded. No transformations are applied at this stage.

In [39]:
# Load data
df = load_data()

Data loaded. Shape: (284807, 31)


### Perform the Three-Way Split

The dataset's extreme class imbalance (~0.17% fraud) is preserved across all three sets via stratification.

In [40]:
# Split
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)


Train: (170883, 30) Validation: (56962, 30) Test: (56962, 30)

Train Target Distribution:
 Target
0    170588
1       295
Name: count, dtype: int64

Validation Target Distribution:
 Target
0    56863
1       99
Name: count, dtype: int64

Test Target Distribution:
 Target
0    56864
1       98
Name: count, dtype: int64


### Preserving Original Transaction Amounts (Validation & Test)

The `Amount` feature is overwritten by scaling in the next step, but Notebook 5's 
cost-sensitive analysis requires the real dollar value of each transaction to 
compute financially accurate misclassification costs. The original amounts for 
validation and test sets are extracted and saved here, before scaling, with reset 
indices to ensure correct row alignment with the corresponding scaled feature files.

In [41]:
# Preserve original (unscaled) transaction amounts AND time values for 
# validation and test sets, needed later for dollar-weighted cost analysis 
# (Notebook 5) and hour-of-day analysis (Notebooks 5 & 6).
# Must be extracted BEFORE scaling, since scaling overwrites the real values.

amount_val = X_val["Amount"].reset_index(drop=True)
amount_test = X_test["Amount"].reset_index(drop=True)
time_val = X_val["Time"].reset_index(drop=True)
time_test = X_test["Time"].reset_index(drop=True)

amount_val.to_csv(os.path.join(PROCESSED_PATH, "amount_val.csv"), index=False, header=["Amount"])
amount_test.to_csv(os.path.join(PROCESSED_PATH, "amount_test.csv"), index=False, header=["Amount"])
time_val.to_csv(os.path.join(PROCESSED_PATH, "time_val.csv"), index=False, header=["Time"])
time_test.to_csv(os.path.join(PROCESSED_PATH, "time_test.csv"), index=False, header=["Time"])

print("Original (unscaled) Amount and Time saved for validation and test sets.")
print("amount_val shape:", amount_val.shape, "| time_val shape:", time_val.shape)
print("amount_test shape:", amount_test.shape, "| time_test shape:", time_test.shape)

Original (unscaled) Amount and Time saved for validation and test sets.
amount_val shape: (56962,) | time_val shape: (56962,)
amount_test shape: (56962,) | time_test shape: (56962,)


### Apply Feature Scaling

The scaler is fit on the training set only and applied to validation and test without refitting.


### note to myself
Standardization is applied using StandardScaler.

Rationale:

• Distance-based methods (e.g., SMOTE, KNN, SVM) require comparable feature scales  
• Features such as Time and Amount vary in magnitude  
• Scaling prevents dominance of high-magnitude variables  

The scaler is fit exclusively on training data and then applied to the test set.

In [42]:
# Scale
X_train_scaled, X_val_scaled, X_test_scaled, scaler = feature_scaling(X_train, X_val, X_test)

Scaling complete.


### Apply SMOTE to Training Data

Oversampling is applied after scaling and restricted to the training set. Validation and test distributions remain untouched, ensuring evaluation reflects real-world class imbalance.

### Note to myself
Given the extreme imbalance, oversampling is applied to the training set using SMOTE.

Important:
• Applied after scaling  
• Applied to training data only  
• Test distribution remains original  

This ensures fair model evaluation without leakage.

In [43]:
# SMOTE (training only)
X_train_resampled, y_train_resampled = handle_class_imbalance(X_train_scaled, y_train)

After SMOTE:
 Target
0    170588
1    170588
Name: count, dtype: int64


### Save Preprocessed Artifacts

The following are saved: SMOTE-resampled training features and labels, scaled validation and test features (original class distribution), and the fitted scaler.

### Note to myself
The following are saved:

• X_train (SMOTE applied)  
• y_train (SMOTE applied)  
• X_test (scaled, original distribution)  
• y_test (original distribution)  
• Fitted scaler  

Additionally, the original scaled training set is preserved for unsupervised modeling experiments.


In [44]:
# Save outputs
save_preprocessed_data(
    X_train_resampled, X_val_scaled, X_test_scaled,
    y_train_resampled, y_val, y_test,
    scaler
)

All preprocessed data (train/val/test) and scaler saved successfully.


### Save Original (Pre-SMOTE) Training Set
The scaled training set is also saved in its original, non-resampled form. This is used for hyperparameter tuning in Notebook 4 — where SMOTE is instead applied inside the cross-validation pipeline rather than beforehand, to prevent synthetic samples from leaking across CV folds — and for the unsupervised anomaly detection notebook, where no resampling is applied at all.

In [45]:
# Also save the original (non-SMOTE) scaled training set for unsupervised modeling — unchanged from before
pd.DataFrame(X_train_scaled).to_csv(f"{PROCESSED_PATH}/X_train_scaled_original.csv", index=False)
pd.DataFrame(y_train, columns=["Target"]).to_csv(f"{PROCESSED_PATH}/y_train_original.csv", index=False)
print("Original scaled training set saved for unsupervised modeling.")

Original scaled training set saved for unsupervised modeling.


## Key Preprocessing Decisions

- Stratification preserves minority-class representation across all three splits
- Scaling is applied before SMOTE, ensuring synthetic samples are generated in a meaningful, comparable feature space
- Oversampling is restricted to the training set only
- Validation and test sets retain their original class distribution throughout

The dataset is now ready for baseline model training.

## Summary
A three-way stratified split (train/validation/test) is used instead of a simple train/test split. The validation set is reserved for all model and threshold selection decisions in downstream notebooks (baseline shortlisting, hyperparameter tuning sanity checks, cost-sensitive threshold selection). The test set is not used for any decision-making and is reserved exclusively for final, one-time performance reporting.

## ------------------------------------------------ END --------------------------------------------------

# PERSONAL
## Modelling
* Logistic Regression gives high interpretability and strong baseline
* SVM & Neural Nets detect complex non-linear patterns
* Tree-based models capture interactions & non-linearities
* Ensemble models like XGBoost often perform best

# Models that require scaling
* Logistic Regression
* SVM
* KNN
* Neural Networks
* PCA-based models

# Models that dont require scaling
* Decision Tree
* Random Forest
* XGBoost
* LightGBM
* CatBoost

## Evaluation metrices for classification problem
* Precision
* Recall
* F1
* ROC AUC
* PR AUC